In [1]:
import numpy as np
import pandas as pd
import math
from numpy.testing import assert_equal

# Problem 2

For this problem, consider each recipe in `/mnt/data/public/food-com-recipes/RAW_recipes.csv` as a transaction and their tags as items in the transaction.

## Problem 2a [10 pts]

Create a function `largest_baskets` that returns the list of all largest baskets of frequent tags in terms of the number of items in the basket. A set of tags is frequent if it is found in at least 25% of the recipes. Sort the baskets by decreasing support. Sort the items in each basket lexicographically.

In [2]:
def get_utility():
    path = "/mnt/data/public/food-com-recipes/RAW_interactions.csv"
    df = pd.read_csv(path, usecols=["user_id", "recipe_id", "rating"])

    # keep only items with >= 250 ratings
    counts = df.groupby("recipe_id")["rating"].count()
    keep = counts[counts >= 250].index
    df = df[df["recipe_id"].isin(keep)]

    # long -> wide: rows = users, columns = items, NaN = not rated
    um = df.pivot_table(index="user_id", columns="recipe_id",
                        values="rating", aggfunc="mean")

    return um.sort_index(axis=0).sort_index(axis=1)

In [3]:
def apriori(db, minsup):
    universe = sorted(set().union(*db))
    tid = {item: {i for i, t in enumerate(db) if item in t} for item in universe}
    k = 1
    frequent = []
    L_prev = None
    while True:
        L_k = []
        if k == 1:
            C_k = [(item,) for item in universe]
        else:
            candidates = set()
            for i in range(len(L_prev)):
                for j in range(i + 1, len(L_prev)):
                    a, b = L_prev[i], L_prev[j]
                    if a[:k-2] == b[:k-2] and a[-1] != b[-1]:
                        candidates.add(tuple(sorted(set(a) | set(b))))
            C_k = sorted(candidates)
        for combo in C_k:
            support = len(set.intersection(*(tid[x] for x in combo)))
            if support >= minsup:
                L_k.append((combo, support))
        if not L_k:
            break
        frequent.extend(L_k)
        L_prev = [c for c, s in L_k]
        k += 1
    return frequent

In [4]:
def largest_baskets():
    likes = get_utility() >= 3
    N = len(likes)
    min_count = math.ceil(round(0.002 * N, 9))
    db = [set(likes.columns[row]) for row in likes.to_numpy()]

    counts = dict(apriori(db, min_count))

    non_max = set()
    for s in counts:
        if len(s) > 1:
            for i in range(len(s)):
                non_max.add(s[:i] + s[i+1:])

    maximal = [s for s in counts if len(s) >= 2 and s not in non_max]
    result = sorted(maximal, key=lambda s: (-counts[s], s))
    return [list(s) for s in result]

In [5]:
lb = largest_baskets()
assert_equal(
    lb[:2],
    [
        [
            "course",
            "dietary",
            "main-ingredient",
            "occasion",
            "preparation",
            "time-to-make",
        ],
        [
            "course",
            "dietary",
            "easy",
            "main-ingredient",
            "preparation",
            "time-to-make",
        ],
    ],
)

AssertionError: 
Items are not equal:
item=0

 ACTUAL: 2
 DESIRED: 6

## Problem 2b [10 pts]

Create a function `highest_lifts` that will return the 10 tag association rules with the highest lift. Each rule should have a support of at least 25% and a confidence of at least 0.8. Return as a tuple of antecedent and consequent. Sort the rules by decreasing lift then by antecedent. Sort the antecedent lexicographically. Consider only 1-itemset consequents.

In [6]:
def highest_lifts():
    n = len(transactions)
    sup_a = abs_support(A, transactions)
    sup_ab = abs_support(A | B, transactions)
    conf = sup_ab / sup_a
    sup_b = abs_support(B, transactions)
    rel_b = sup_b / n
    return conf / rel_b

In [7]:
hl = highest_lifts()
assert_equal(
    hl[:5],
    [
        (["course", "low-in-something"], "dietary"),
        (["course", "low-in-something", "main-ingredient"], "dietary"),
        (
            ["course", "low-in-something", "main-ingredient", "preparation"],
            "dietary",
        ),
        (
            [
                "course",
                "low-in-something",
                "main-ingredient",
                "preparation",
                "time-to-make",
            ],
            "dietary",
        ),
        (
            ["course", "low-in-something", "main-ingredient", "time-to-make"],
            "dietary",
        ),
    ],
)

NameError: name 'transactions' is not defined